In [1]:
import json
import pandas as pd
import re

In [2]:
def clean_text(text):
    """Removes excessive newlines and normalizes spaces."""
    if pd.isna(text) or text is None:
        return ""
    
    text = str(text)
    # Replace newlines and excessive whitespace with a single space
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def clean_pregnancy_column(text):
    """Cleans up the structured array look of pregnancy data."""
    if pd.isna(text) or text is None:
        return ""
    text = str(text)
    # Simple cleanup to make it more natural readable text
    text = text.replace("['", "").replace("']", "").replace("'", "")
    return text

def process_data():
    print("Loading datasets...")
    # Load JSON files
    with open('French/Medicine.json', 'r', encoding='utf-8') as f:
        medicine_df = pd.DataFrame(json.load(f))
    
    with open('French/Leaflet.json', 'r', encoding='utf-8') as f:
        leaflet_df = pd.DataFrame(json.load(f))
        
    print("Merging datasets on cisCode...")
    # Merge both JSON files on cisCode
    merged_df = pd.merge(medicine_df, leaflet_df, on='cisCode', how='left')
    
    print("Cleaning text columns...")
    text_columns = [
        'indications', 'posology', 'contraindicationsAndWarnings', 
        'adverseEffects', 'interactions', 'mechanismOfAction'
    ]
                    
    for col in text_columns:
        if col in merged_df.columns:
            merged_df[col] = merged_df[col].apply(clean_text)
            
    if 'pregnancyBreastfeeding' in merged_df.columns:
        merged_df['pregnancyBreastfeeding'] = merged_df['pregnancyBreastfeeding'].apply(clean_pregnancy_column)
        
    # Optional: Drop items entirely if they have no useful RAG textual data
    # (Leaving this out initially depending on how strictly you want to prune)
    # merged_df = merged_df[(merged_df['indications'] != "") | (merged_df['mechanismOfAction'] != "")]
    
    # Create the concatenated Full Text column designed specially to be embedded for RAG
    merged_df['rag_content'] = merged_df.apply(lambda row: f"""
Nom du médicament: {row.get('brandName', '')}
DCI (Principe actif): {row.get('dci', '')}
Forme: {row.get('form', '')}
Indications: {row.get('indications', 'Non précisé')}
Posologie: {row.get('posology', 'Non précisé')}
Effets secondaires: {row.get('adverseEffects', 'Non précisé')}
Mécanisme d'action: {row.get('mechanismOfAction', 'Non précisé')}
Interactions: {row.get('interactions', 'Non précisé')}
""".strip(), axis=1)

    print("Saving cleaned data...")
    # Save cleaned data to a new CSV ready to be embedded into Vector DB
    output_path = 'French/Cleaned_Data_For_RAG.csv'
    merged_df.to_csv(output_path, index=False, encoding='utf-8')
    print(f"Done! Cleaned output saved to {output_path}")


In [3]:
if __name__ == "__main__":
    process_data()

Loading datasets...
Merging datasets on cisCode...
Cleaning text columns...
Saving cleaned data...
Done! Cleaned output saved to French/Cleaned_Data_For_RAG.csv


In [4]:
df = pd.read_csv('French/Cleaned_Data_For_RAG.csv')
df.head()

,cisCode,source,url,brandName,dci,form,indications,posology,contraindicationsAndWarnings,adverseEffects,interactions,pregnancyBreastfeeding,conservation,mechanismOfAction,rag_content
0,24 D 004,vidal,https://www.vidal.fr/medicaments/desferal-500-...,FROXA,Mesilate De Deferoxamine,poudre,Déféroxamine mésilate 500 mg poudre et solvant...,Posologie Unité de prise flacon déféroxamine m...,NaN,ANOMALIE DES EXAMENS DE LABORATOIRE: Hypocalcé...,Les informations fournies sur les interactions...,"Pregnancy: II, II, Breastfeeding: II",Cip : 3400930298374 Modalités de conservation ...,Déféroxamine : Mécanisme d'action La déféroxam...,Nom du médicament: FROXA\nDCI (Principe actif)...
1,05 D 112,vidal,https://base-donnees-publique.medicaments.gouv...,TRAZIMERA,Trastuzumab,poudre,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Nom du médicament: TRAZIMERA\nDCI (Principe ac...
2,10 F 147,vidal,https://www.vidal.fr/medicaments/metopimazine-...,VOMICED,Metopimazine,comprimé,"Métopimazine 7,5 mg comprimé orodispersible De...",Posologie Unité de prise comprimé métopimazine...,NaN,ANOMALIE DES EXAMENS DE LABORATOIRE: DERMATOLO...,Les informations fournies sur les interactions...,"Pregnancy: II, II, Breastfeeding: II",Cip : 3400930274095 Modalités de conservation ...,Métopimazine : Mécanisme d'action La métopimaz...,Nom du médicament: VOMICED\nDCI (Principe acti...
3,12 J 182,vidal,https://base-donnees-publique.medicaments.gouv...,CLOTTAFACT,Fibrinogene Humain,poudre,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Nom du médicament: CLOTTAFACT\nDCI (Principe a...
4,09 H 031,vidal,https://www.vidal.fr/medicaments/methylprednis...,LISAMETHYLE,Methylprednisolone (Sous Forme De Succinate So...,Pdre. et solv. p. sol. inj. im/iv,Méthylprednisolone (hémisuccinate) 500 mg poud...,Posologie Unité de prise flacon méthylpredniso...,NaN,ANOMALIE DES EXAMENS DE LABORATOIRE: Phosphata...,Les informations fournies sur les interactions...,"Pregnancy: II, II, Breastfeeding: II",Cip : 3400935541017 Modalités de conservation ...,Méthylprednisolone : Mécanisme d'action Les gl...,Nom du médicament: LISAMETHYLE\nDCI (Principe ...


In [1]:
from pathlib import Path

english_dir = Path("English")

if not english_dir.exists():
    print("English folder not found.")
else:
    print("JSON count per subdirectory:\n")
    total = 0

    # each subdirectory under English (including nested folders)
    for folder in sorted([p for p in english_dir.rglob("*") if p.is_dir()]):
        count = sum(1 for f in folder.glob("*.json"))
        if count > 0:
            rel = folder.relative_to(english_dir)
            print(f"{rel}: {count}")
            total += count

    print(f"\nTotal JSON files found: {total}")

JSON count per subdirectory:

not_found: 1044
success: 2775

Total JSON files found: 3819
